# SOLID Principles — Complete Python Study Notebook

**Category:** Software Design & Architecture  
**Level:** Foundation → Intermediate  
**Language:** Python 3.11+  
**Prerequisites:** Python OOP, classes, inheritance, composition, type hints, and basic testing.

> **Notebook structure:** This is a real Jupyter notebook with separate, hierarchical **Markdown cells for explanations** and **executable Python cells for examples**. It is not one large Markdown cell. Run code cells from top to bottom.

**Learning method:** For each principle, first inspect the problematic design, predict its consequences, execute the demonstration, study the refactoring, then complete the checkpoint and exercises. Not all code needs all five principles; optimize for clear, appropriate design.

## Table of contents

1. [Introduction to SOLID](#1-introduction-to-solid)
2. [Single Responsibility Principle (SRP)](#2-single-responsibility-principle-srp)
3. [Open/Closed Principle (OCP)](#3-openclosed-principle-ocp)
4. [Liskov Substitution Principle (LSP)](#4-liskov-substitution-principle-lsp)
5. [Interface Segregation Principle (ISP)](#5-interface-segregation-principle-isp)
6. [Dependency Inversion Principle (DIP)](#6-dependency-inversion-principle-dip)
7. [Dependency Inversion vs Dependency Injection](#7-dependency-inversion-vs-dependency-injection)
8. [How the Principles Work Together](#8-how-the-five-principles-work-together)
9. [SOLID in Real-World Architecture](#9-solid-in-real-world-software-architecture)
10. [Common Mistakes and Overengineering](#10-common-mistakes-and-overengineering)
11. [Testing SOLID Designs](#11-testing-solid-designs)
12. [Practical Exercises](#12-practical-exercises)
13. [Mastery Checklist](#13-mastery-checklist)
14. [Recommended Resources](#14-recommended-learning-resources)

A **modular version** is also supplied: one `.ipynb` per principle, with numbered filenames. Complete them in sequence.

# 1. Introduction to SOLID

SOLID comprises five object-oriented design principles associated with Robert C. Martin. They help manage change by improving the organization of responsibilities, extension points, behavioral contracts, client interfaces and dependency direction.

| Principle | Main question | Main risk it addresses |
|---|---|---|
| **S — Single Responsibility** | Does this component have one coherent reason to change? | Mixed unrelated responsibilities |
| **O — Open/Closed** | Can expected new behavior be added by extension? | Repeatedly editing stable conditional logic |
| **L — Liskov Substitution** | Can implementations replace one another without surprising clients? | Broken behavioral contracts |
| **I — Interface Segregation** | Do clients depend only on capabilities they actually use? | Fat interfaces and unsupported operations |
| **D — Dependency Inversion** | Do business policies rely on stable abstractions rather than concrete infrastructure? | Tight coupling to databases, APIs or frameworks |

**Why they matter:** As applications gain payments, notifications, persistence, authentication and integrations, unrelated changes can become entangled. SOLID is one set of tools for reducing this cost—not a mandate to create more classes.

**Important distinctions:** SRP is not one-method-per-class; OCP is not 'never edit existing code'; LSP concerns behavior, not only type signatures; ISP is not one-method-per-interface; and DIP is not identical to dependency injection.

# 2. Single Responsibility Principle (SRP)

## 2.1 Definition

> A module should have one reason to change.

A practical interpretation: a component should be responsible to one actor or group of stakeholders whose decisions may force it to change. One coherent responsibility can require many closely related methods. Separate independent reasons to change, rather than mechanically creating one class per method.

### Example: three different reasons to change

An invoice workflow might change when:

- Finance revises tax or pricing calculations.
- Engineering moves storage from SQLite to PostgreSQL.
- Customer operations changes email templates or providers.

Putting all three concerns in one class ties independent changes together.

## 2.2 Violation: a class with unrelated concerns

Inspect this example. Its business logic, database access and messaging share one class. For reproducibility, this demonstration uses an in-memory SQLite database and a print-based notification.

In [ ]:
import sqlite3

class BadInvoice:
    def __init__(self, customer_email: str, items: list[dict]):
        self.customer_email = customer_email
        self.items = items

    def calculate_total(self) -> float:
        return sum(item["price"] * item["quantity"] for item in self.items)

    def save_to_database(self, connection: sqlite3.Connection) -> None:
        connection.execute("CREATE TABLE IF NOT EXISTS invoices (email TEXT, total REAL)")
        connection.execute("INSERT INTO invoices VALUES (?, ?)",
                           (self.customer_email, self.calculate_total()))
        connection.commit()

    def send_email(self) -> None:
        print(f"Sending invoice to {self.customer_email}")

bad_invoice = BadInvoice("customer@example.com", [{"price": 100, "quantity": 2}])
with sqlite3.connect(":memory:") as connection:
    bad_invoice.save_to_database(connection)
    print("Stored total:", connection.execute("SELECT total FROM invoices").fetchone()[0])
bad_invoice.send_email()

### Identify the problems

- **Multiple change drivers:** pricing, persistence and messaging independently affect `BadInvoice`.
- **Testing friction:** checking total calculation can be entangled with infrastructure concerns as the code evolves.
- **Fragility:** modifying unrelated features requires revisiting the same class.
- **Low cohesion:** a database schema migration is not a responsibility of an invoice domain entity.

Separating responsibilities does **not** mean that orchestration is forbidden. A service may legitimately coordinate several steps if workflow coordination is its coherent responsibility.

## 2.3 Refactoring SRP — step 1: domain model

The invoice model owns invoice data and invoice-related rules. `Decimal` is preferred to binary floating-point for monetary amounts.

In [ ]:
from dataclasses import dataclass
from decimal import Decimal

@dataclass(frozen=True)
class InvoiceItem:
    unit_price: Decimal
    quantity: int

    def __post_init__(self):
        if self.unit_price < 0 or self.quantity < 0:
            raise ValueError("Price and quantity must be nonnegative")

@dataclass(frozen=True)
class Invoice:
    customer_email: str
    items: tuple[InvoiceItem, ...]

    def calculate_total(self) -> Decimal:
        return sum((item.unit_price * item.quantity for item in self.items),
                   start=Decimal("0.00"))

invoice = Invoice("customer@example.com", (
    InvoiceItem(Decimal("100.00"), 2),
    InvoiceItem(Decimal("50.00"), 1),
))
assert invoice.calculate_total() == Decimal("250.00")
print("Invoice total:", invoice.calculate_total())

## 2.4 Refactoring SRP — step 2: persistence

The repository owns database operations. Changing schema or storage technology should not force changes to the domain model.

In [ ]:
class InvoiceRepository:
    def __init__(self, connection: sqlite3.Connection):
        self.connection = connection
        self.connection.execute("""
            CREATE TABLE IF NOT EXISTS invoices (
                email TEXT NOT NULL,
                total TEXT NOT NULL
            )
        """)

    def save(self, invoice: Invoice) -> None:
        self.connection.execute(
            "INSERT INTO invoices (email, total) VALUES (?, ?)",
            (invoice.customer_email, str(invoice.calculate_total()))
        )
        self.connection.commit()

## 2.5 Refactoring SRP — step 3: notification and workflow

The sender owns delivery. The application service orchestrates saving and notifying, without knowing SQL or SMTP details. Here the sender is a demonstration adapter, not a real email integration.

In [ ]:
class InvoiceEmailSender:
    def send(self, invoice: Invoice) -> None:
        print(f"Invoice notification: {invoice.customer_email}")

class InvoiceService:
    def __init__(self, repository: InvoiceRepository, sender: InvoiceEmailSender):
        self.repository = repository
        self.sender = sender

    def process(self, invoice: Invoice) -> None:
        self.repository.save(invoice)
        self.sender.send(invoice)

with sqlite3.connect(":memory:") as connection:
    repository = InvoiceRepository(connection)
    service = InvoiceService(repository, InvoiceEmailSender())
    service.process(invoice)
    row = connection.execute("SELECT email, total FROM invoices").fetchone()
    assert row == ("customer@example.com", "250.00")
    print("Verified persisted invoice:", row)

## 2.6 SRP vs Separation of Concerns

**Separation of Concerns (SoC)** is a broad principle: isolate different aspects of a system. **SRP** makes the question more precise by focusing on coherent responsibilities and reasons to change. Separating the API, business rules and database is SoC; organizing each component around one coherent change driver is SRP.

## 2.7 Trade-offs and anti-patterns

**Don't** introduce `InvoiceAdder`, `InvoiceTaxAdder`, `InvoiceTotalCalculator`, etc. simply to reduce method counts. Splitting closely related invoice operations across tiny classes can harm readability and cohesion. Also account for transaction boundaries: a real notification workflow may need an outbox rather than an immediate email after committing an invoice.

### SRP checkpoint

- [ ] Can I name the responsibility of each class in one specific sentence?
- [ ] Which stakeholder-driven change would alter this component?
- [ ] Does a database migration force unrelated business-code changes?
- [ ] Are independently changing details isolated?
- [ ] Can I explain why every split reduces rather than increases complexity?

# 3. Open/Closed Principle (OCP)

## 3.1 Definition

> Software entities should be open for extension but closed for modification.

For known or likely sources of variation, new behavior should be introducible by extending an existing contract instead of repeatedly modifying stable clients. OCP is **not** a ban on changing existing code: bug fixes, changing requirements and refactoring remain legitimate.

**Scenario:** An e-commerce application introduces new payment providers over time.

## 3.2 Violation: branching on payment type

Every new provider forces changes to the same processing method, creating repeated edits and increasing regression risk.

In [ ]:
class ConditionalPaymentProcessor:
    def process(self, method: str, amount: float) -> None:
        if method == "credit_card":
            print("Credit card:", amount)
        elif method == "paypal":
            print("PayPal:", amount)
        elif method == "bank_transfer":
            print("Bank transfer:", amount)
        else:
            raise ValueError("Unknown payment type")

ConditionalPaymentProcessor().process("paypal", 100.0)

## 3.3 Refactoring: define a stable strategy contract

Use a focused abstraction for the variation point. These examples print rather than transfer real money. For real payment systems, model amounts using `Decimal` or integer minor units and specify idempotency and failure behavior.

In [ ]:
from abc import ABC, abstractmethod
from decimal import Decimal

class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount: Decimal) -> None:
        """Process nonnegative amount or raise a documented error."""

class CreditCardPayment(PaymentMethod):
    def pay(self, amount: Decimal) -> None:
        print("Credit card:", amount)

class PayPalPayment(PaymentMethod):
    def pay(self, amount: Decimal) -> None:
        print("PayPal:", amount)

class BankTransferPayment(PaymentMethod):
    def pay(self, amount: Decimal) -> None:
        print("Bank transfer:", amount)

class PaymentProcessor:
    def process(self, payment: PaymentMethod, amount: Decimal) -> None:
        if amount < 0:
            raise ValueError("Amount must be nonnegative")
        payment.pay(amount)

processor = PaymentProcessor()
processor.process(CreditCardPayment(), Decimal("150.00"))
processor.process(PayPalPayment(), Decimal("200.00"))

## 3.4 Add a new provider without editing the processor

A new strategy implements the existing contract. The **composition root** still decides which provider to select—OCP does not eliminate configuration changes.

In [ ]:
class ApplePayPayment(PaymentMethod):
    def pay(self, amount: Decimal) -> None:
        print("Apple Pay:", amount)

processor.process(ApplePayPayment(), Decimal("300.00"))

## 3.5 Why this resembles the Strategy Pattern

The strategy encapsulates interchangeable behavior behind a common contract. Other variation points: discounts, shipping prices, file exporters, authentication mechanisms and compression algorithms. Strategy is one way to apply OCP; OCP itself is broader than any one design pattern.

## 3.6 When an abstraction is premature

If an application has one payment method and there is no plausible variation, adding a large inheritance hierarchy may make the code harder to understand. Start simple, protect behavior with tests, and introduce the extension point when changes demonstrate its value. Balance OCP against **YAGNI** (You Aren't Gonna Need It) and **KISS** (Keep It Simple).

### OCP checkpoint

- [ ] Which variation is the design intended to accommodate?
- [ ] Can I add a new implementation without rewriting the stable business workflow?
- [ ] Did I move the repeated `if/elif` branching to an appropriate composition or registration point?
- [ ] Is the abstraction cohesive and stable?
- [ ] Can I justify its cost today?

# 4. Liskov Substitution Principle (LSP)

## 4.1 Definition

> Implementations of a type should be usable in place of that type without violating their clients' behavioral expectations.

Matching method signatures is **not enough**. A meaningful contract includes accepted inputs, guarantees, invariants, side effects, failure behavior and sometimes ordering, latency or consistency assumptions.

### Behavioral obligations

Generally, a subtype must not:

- **Strengthen preconditions:** reject inputs its base contract accepts.
- **Weaken postconditions:** provide fewer guarantees after successful calls.
- **Break invariants:** violate properties promised by the base abstraction.
- **Introduce incompatible behavior:** surprise code that relies on documented semantics.

## 4.2 Violation: a shipping subtype strengthens a precondition

The base contract promises to handle **every nonnegative weight**. Standard shipping honors that promise; the first express implementation does not.

In [ ]:
from abc import ABC, abstractmethod

class ShippingMethod(ABC):
    @abstractmethod
    def calculate_cost(self, weight: float) -> float:
        """Accept all weights >= 0 and return a nonnegative cost."""

class StandardShipping(ShippingMethod):
    def calculate_cost(self, weight: float) -> float:
        if weight < 0:
            raise ValueError("Weight cannot be negative")
        return weight * 5.0

class BadExpressShipping(ShippingMethod):
    def calculate_cost(self, weight: float) -> float:
        if weight < 1:
            raise ValueError("Minimum weight is 1 kg")
        return weight * 10.0

def quote_half_kilo(shipping: ShippingMethod) -> float:
    return shipping.calculate_cost(0.5)

print("Standard:", quote_half_kilo(StandardShipping()))
try:
    print("Bad express:", quote_half_kilo(BadExpressShipping()))
except ValueError as error:
    print("Expected contract violation:", error)

## 4.3 Repair the implementation **or** repair the abstraction

If express shipping genuinely accepts every nonnegative weight, fix its implementation. If the business truly forbids some weights, change the model or contract instead. Do not lie about the business domain merely to satisfy inheritance.

In [ ]:
class ExpressShipping(ShippingMethod):
    def calculate_cost(self, weight: float) -> float:
        if weight < 0:
            raise ValueError("Weight cannot be negative")
        return 15.0 + weight * 10.0

assert quote_half_kilo(StandardShipping()) == 2.5
assert quote_half_kilo(ExpressShipping()) == 20.0
print("Both implementations preserve the accepted input domain")

## 4.4 Preconditions, postconditions and invariants

- **Precondition:** a condition a caller must meet, e.g. nonnegative weight.
- **Postcondition:** a guaranteed result when the operation succeeds, e.g. cost is nonnegative.
- **Invariant:** a property preserved by valid operations, e.g. a non-overdraft account has balance ≥ 0.

A repository contract might promise that a successful `save(record)` makes that record retrievable. An implementation that silently discards data violates the behavioral contract even if its signature matches.

### Exception behavior

A subtype that raises `NotImplementedError` for a valid operation is not a compatible implementation of a fully supported contract. Not every new exception is an LSP violation; assess the contract and explicitly documented failures.

### If a genuine restriction exists

Options include checking service eligibility separately, introducing capability-specific contracts, or documenting a uniform rejection mechanism in the shared contract. These are **domain-modeling decisions**, not mechanical inheritance tricks.

## 4.5 Shared contract tests

Run the same behavioral tests against every compatible implementation. The first test checks positive inputs and outputs; the second checks rejection of invalid negative weights.

In [ ]:
def shipping_contract_test(factory):
    shipping = factory()
    for weight in (0, 0.5, 2.0):
        result = shipping.calculate_cost(weight)
        assert result >= 0, (factory.__name__, weight, result)
    try:
        shipping.calculate_cost(-1)
    except ValueError:
        pass
    else:
        raise AssertionError(f"{factory.__name__} accepted invalid negative weight")

for factory in (StandardShipping, ExpressShipping):
    shipping_contract_test(factory)
print("Contract tests passed for StandardShipping and ExpressShipping")

### LSP checkpoint

- [ ] Can I state the abstraction's preconditions, postconditions and invariants?
- [ ] Do all implementations accept the required input domain?
- [ ] Does every implementation preserve promised success and failure semantics?
- [ ] Can clients switch implementations without hidden special cases?
- [ ] Did I use shared contract tests rather than testing only one implementation?

# 5. Interface Segregation Principle (ISP)

## 5.1 Definition

> Clients should not be forced to depend on methods they do not use.

Interfaces should describe cohesive capabilities **from the client's perspective**. A fat interface can force implementations to expose nonsensical operations, and make unrelated changes propagate to consumers.

## 5.2 Violation: a one-size-fits-all office machine

A basic printer can print, but cannot scan or fax. Forcing it to inherit an interface for all three capabilities creates a misleading public API.

In [ ]:
from abc import ABC, abstractmethod

class BadOfficeMachine(ABC):
    @abstractmethod
    def print_document(self, document: str) -> None: ...
    @abstractmethod
    def scan_document(self) -> str: ...
    @abstractmethod
    def send_fax(self, number: str) -> None: ...

class BadSimplePrinter(BadOfficeMachine):
    def print_document(self, document: str) -> None:
        print("Printing:", document)
    def scan_document(self) -> str:
        raise NotImplementedError("This printer cannot scan")
    def send_fax(self, number: str) -> None:
        raise NotImplementedError("This printer cannot fax")

bad_printer = BadSimplePrinter()
bad_printer.print_document("Report")
try:
    bad_printer.scan_document()
except NotImplementedError as error:
    print("Expected ISP problem:", error)

## 5.3 Refactoring: capability-focused Protocols

Python `Protocol` uses structural subtyping: a class satisfies the contract by having compatible methods; explicit inheritance isn't required. Static tools such as mypy or Pyright can check compatibility.

In [ ]:
from typing import Protocol

class Printer(Protocol):
    def print_document(self, document: str) -> None: ...

class Scanner(Protocol):
    def scan_document(self) -> str: ...

class Fax(Protocol):
    def send_fax(self, number: str) -> None: ...

class SimplePrinter:
    def print_document(self, document: str) -> None:
        print("Printing:", document)

class MultiFunctionPrinter:
    def print_document(self, document: str) -> None:
        print("Printing:", document)
    def scan_document(self) -> str:
        return "Scanned document"
    def send_fax(self, number: str) -> None:
        print("Fax to:", number)

def print_report(printer: Printer, report: str) -> None:
    printer.print_document(report)

def scan_report(scanner: Scanner) -> str:
    return scanner.scan_document()

simple = SimplePrinter()
multi = MultiFunctionPrinter()
print_report(simple, "Annual Report")
print_report(multi, "Annual Report")
assert scan_report(multi) == "Scanned document"

## 5.4 Protocol vs ABC

**Protocol:** expressive structural contracts, especially useful for application ports and testing adapters. Runtime compatibility is not automatically enforced just by adding annotations.

**ABC (`abc.ABC`):** explicit inheritance, abstract methods and optionally shared implementation. Useful when an explicit class hierarchy provides value.

A focused interface can legitimately contain multiple related methods. Transaction management, for example, may need `begin`, `commit` and `rollback` as one coherent capability. ISP does **not** demand one method per interface.

### ISP checkpoint

- [ ] Does every method on an interface serve its clients?
- [ ] Do some implementations raise `NotImplementedError` for normal operations?
- [ ] Can read-only clients avoid write/delete dependencies?
- [ ] Would a more focused contract simplify testing and reuse?
- [ ] Would further splitting actually hurt cohesion?

# 6. Dependency Inversion Principle (DIP)

## 6.1 Definition

1. High-level policy modules should not depend directly on low-level implementation modules; both should depend on abstractions.
2. Abstractions should not depend on implementation details; details should depend on abstractions.

A **high-level policy** implements a business rule or workflow. A **low-level implementation** might be MySQL, SMTP, an HTTP client or a filesystem. DIP controls *source-code dependency direction*, not merely how objects are created.

## 6.2 Violation: business logic constructs its concrete database

In [ ]:
class MySQLDatabase:
    def save_order(self, order: dict) -> None:
        print("Saving to MySQL:", order)

class BadOrderService:
    def __init__(self):
        self.database = MySQLDatabase()
    def place_order(self, order: dict) -> None:
        self.database.save_order(order)

BadOrderService().place_order({"order_id": 101, "total": 500})

`BadOrderService` is difficult to reuse with PostgreSQL or an in-memory test double, because it directly constructs and refers to `MySQLDatabase`. A database change may force an application-service change.

## 6.3 Refactoring: define the contract around business needs

In [ ]:
from typing import Protocol

class OrderRepository(Protocol):
    def save(self, order: dict) -> None:
        """Persist an order or signal failure."""

class MySQLOrderRepository:
    def save(self, order: dict) -> None:
        print("MySQL adapter saved:", order)

class PostgreSQLOrderRepository:
    def save(self, order: dict) -> None:
        print("PostgreSQL adapter saved:", order)

class OrderService:
    def __init__(self, repository: OrderRepository):
        self.repository = repository
    def place_order(self, order: dict) -> None:
        self.repository.save(order)

OrderService(MySQLOrderRepository()).place_order({"order_id": 102})
OrderService(PostgreSQLOrderRepository()).place_order({"order_id": 103})

### Dependency direction

**Before:** `OrderService → MySQLDatabase`

**After:** `OrderService → OrderRepository ← MySQLOrderRepository`

The contract typically lives in the domain or application boundary, where the high-level policy can own what it needs. Concrete infrastructure adapts to that contract. This is the **Ports and Adapters** idea in miniature.

## 6.4 Testing with a fake implementation

A fake repository records interactions without running a real database. It tests business-service orchestration; separate adapter integration tests remain necessary.

In [ ]:
class FakeOrderRepository:
    def __init__(self):
        self.saved_orders: list[dict] = []
    def save(self, order: dict) -> None:
        self.saved_orders.append(order)

def test_place_order():
    fake = FakeOrderRepository()
    service = OrderService(fake)
    order = {"order_id": 101, "total": 500}
    service.place_order(order)
    assert fake.saved_orders == [order]

test_place_order()
print("OrderService unit test passed without a database")

## 6.5 Limits of DIP

A protocol for every tiny helper creates needless indirection. Prioritize dependencies at important boundaries (databases, external services, I/O, frameworks, time sources) and where replacement or test isolation has value. An adapter can conform to a Python Protocol structurally, without importing the Protocol into its own source file. Depend on **useful** abstractions, not an abstraction merely because a concrete class exists.

### DIP checkpoint

- [ ] Does the high-level policy import or construct a concrete database or API client?
- [ ] Is the contract defined by what the business layer needs?
- [ ] Can I supply a fake without editing business code?
- [ ] Can I replace concrete adapters at the application boundary?
- [ ] Do I have integration tests for each real adapter?

# 7. Dependency Inversion vs Dependency Injection

## 7.1 They are related, but not identical

**DIP** is an architectural/design principle about source-code dependencies. **Dependency Injection (DI)** is a technique for supplying dependencies externally. DI can be used while still depending on a concrete infrastructure type; that is *not automatically DIP*.

For example, `__init__(sender: ConcreteSMTPClient)` injects a dependency but still refers to a concrete SMTP implementation.

## 7.2 Python example: Protocol + constructor injection

In [ ]:
from typing import Protocol

class MessageSender(Protocol):
    def send(self, recipient: str, message: str) -> None: ...

class EmailSender:
    def send(self, recipient: str, message: str) -> None:
        print(f"Email to {recipient}: {message}")

class NotificationService:
    def __init__(self, sender: MessageSender):
        self.sender = sender
    def notify_user(self, recipient: str) -> None:
        self.sender.send(recipient, "Your order has shipped.")

def main() -> None:
    # Composition root: construct concrete adapters and connect the app.
    sender = EmailSender()
    service = NotificationService(sender)
    service.notify_user("customer@example.com")

main()

## 7.3 Composition root

A **composition root** is a designated place where concrete implementations are created and connected. In a small application it may simply be `main()`; in a large application a dependency-injection container may help manage lifetimes and complex dependency graphs. Frameworks are optional.

A practical design question: *Where will we decide which database, payment gateway or notifier is actually used?* Usually, keep that selection at an entry point rather than scattering it through business logic.

# 8. How the Five Principles Work Together

Consider an order workflow that must save an order, charge through a payment provider and notify a customer.

```text
                      OrderService
                           |
           +---------------+----------------+
           |               |                |
           v               v                v
     OrderRepository   PaymentGateway    MessageSender
        (port)            (port)            (port)
           ^               ^                ^
           |               |                |
      SQLiteAdapter     DemoPayment       DemoNotifier
```

- **SRP:** each adapter owns one infrastructure concern; `OrderService` coordinates the order workflow.
- **OCP:** additional repositories, gateways and senders can implement existing ports.
- **LSP:** replacements must preserve each port's behavioral promises.
- **ISP:** the service sees only required operations.
- **DIP:** workflow policy depends on contracts, not vendor clients.

## 8.1 An executable miniature design

This example is illustrative. A production order system must also define idempotency, transaction boundaries, payment reconciliation, retries and outbox/event delivery where appropriate.

In [ ]:
from dataclasses import dataclass
from decimal import Decimal
from typing import Protocol

@dataclass(frozen=True)
class Order:
    order_id: int
    customer_email: str
    amount: Decimal

class OrderPort(Protocol):
    def save(self, order: Order) -> None: ...

class PaymentPort(Protocol):
    def charge(self, order: Order) -> None: ...

class MessagePort(Protocol):
    def send_confirmation(self, order: Order) -> None: ...

class InMemoryOrders:
    def __init__(self):
        self.saved: list[Order] = []
    def save(self, order: Order) -> None:
        self.saved.append(order)

class DemoPayment:
    def __init__(self):
        self.charged: list[int] = []
    def charge(self, order: Order) -> None:
        if order.amount < 0:
            raise ValueError("Negative order amount")
        self.charged.append(order.order_id)

class DemoNotifier:
    def __init__(self):
        self.sent: list[int] = []
    def send_confirmation(self, order: Order) -> None:
        self.sent.append(order.order_id)

class IntegratedOrderService:
    def __init__(self, repository: OrderPort, payments: PaymentPort, messages: MessagePort):
        self.repository = repository
        self.payments = payments
        self.messages = messages
    def place(self, order: Order) -> None:
        self.payments.charge(order)
        self.repository.save(order)
        self.messages.send_confirmation(order)

repo, payment, notifier = InMemoryOrders(), DemoPayment(), DemoNotifier()
app = IntegratedOrderService(repo, payment, notifier)
order = Order(101, "customer@example.com", Decimal("120.00"))
app.place(order)
assert repo.saved == [order]
assert payment.charged == [101]
assert notifier.sent == [101]
print("Integrated example passed")

## 8.2 Review questions

1. Which responsibilities are independent, and which belong together in workflow coordination?
2. What expected variations justify ports?
3. What success/failure guarantees must *every* payment implementation preserve?
4. Are the ports minimal from `IntegratedOrderService`'s perspective?
5. Which integration tests and failure-handling designs are still missing?

# 9. SOLID in Real-World Software Architecture

## 9.1 Backend / FastAPI

One possible project layout:

```text
api/orders.py                   # HTTP boundary
domain/order.py                 # entities and business rules
application/order_service.py   # workflow
ports/order_repository.py      # persistence contract
ports/payment_gateway.py       # payment contract
infrastructure/postgres.py     # adapter
infrastructure/payment.py      # external payment adapter
```

This separation can improve SRP and DIP, but small projects do not always need so many layers. Choose boundaries based on real change drivers and requirements.

## 9.2 AI systems and RAG

Treat document loaders, chunking strategies, embedding providers, vector stores, retrieval and generation as potentially independent concerns. DIP can isolate external model providers, OCP can introduce additional strategies, and LSP forces attention to embedding dimension, normalization, model version and output semantics—not merely a shared `embed(text)` signature.

In [ ]:
from typing import Protocol

class EmbeddingProvider(Protocol):
    def embed(self, text: str) -> list[float]: ...

class ToyEmbedding:
    """Deterministic demonstration only; not a semantic embedding model."""
    def embed(self, text: str) -> list[float]:
        return [float(len(text)), float(len(text.split()))]

class IndexingService:
    def __init__(self, embeddings: EmbeddingProvider):
        self.embeddings = embeddings
    def index(self, text: str) -> list[float]:
        return self.embeddings.embed(text)

assert IndexingService(ToyEmbedding()).index("SOLID principles") == [16.0, 2.0]

## 9.3 Robotics / embedded systems

A high-level controller may depend on a motor-control protocol, with a simulator and hardware driver as adapters. LSP requires consistent units, speed constraints, stop semantics and error signals. Hard real-time and safety requirements may need stronger guarantees and explicit validation—SOLID alone is not a safety architecture.

## 9.4 Data engineering

Separate extraction, transformation, validation and loading where they change independently. A small `read_records()` port might support CSV, SQL and REST adapters. However, production pipelines might require streaming, schemas, retries, checkpointing and memory limits; contracts must accurately represent those requirements.

# 10. Common Mistakes and Overengineering

| Mistake | Why it is harmful | Better approach |
|---|---|---|
| One class per method | Excessive fragmentation and navigation | Organize around coherent reasons to change |
| Interface for every class | Indirection without a useful boundary | Abstract real variability or important dependencies |
| Inheritance purely to reuse code | Unwanted substitutability obligations | Prefer composition when there is no real subtype relationship |
| Treating OCP as 'never edit code' | Discourages necessary fixes/refactoring | Stabilize *appropriate* variation points |
| Matching signatures = LSP | Ignores invariants and error behavior | Specify and test behavioral contracts |
| One method per interface | Creates unnecessary fragmentation | Group cohesive client capabilities |
| Huge dependency-injection framework | Complexity out of proportion to app size | Start with constructor injection and a `main()` composition root |

## Design trade-off exercise

A feature has one implementation now and a speculative possibility of three future variants. Compare:

1. One simple implementation with unit tests.
2. A strategy interface and one concrete strategy.
3. A plugin registry with dynamic discovery.

Which is simplest *given today's evidence*? What concrete new requirement would justify moving from option 1 to 2 or 3? Document both maintenance costs and expected benefits.

The goal is to **reduce overall system complexity**, not maximize interfaces, classes or design patterns.

# 11. Testing SOLID Designs

Tests support SOLID, but passing tests does not prove a design has good responsibilities or useful abstractions. Combine behavioral tests with architecture reviews.

- **SRP:** test coherent responsibilities separately (calculations, persistence, messaging and orchestration).
- **OCP:** add tests for a new implementation and rerun tests for unchanged clients.
- **LSP:** execute one contract-test suite against every implementation.
- **ISP:** a print-only client should not require scan/fax test doubles.
- **DIP:** use fake dependencies for business tests, and real adapter integration tests for infrastructure.

### Why both unit and integration tests?

A fake repository may not reproduce real transaction, locking, error or consistency behavior. Business-service tests are necessary but not sufficient; real adapters need independent integration tests.

## 11.1 Standalone executable contract-test example

In [ ]:
class InMemoryKeyValueStore:
    def __init__(self):
        self.records = {}
    def save(self, key: str, value: str) -> None:
        self.records[key] = value
    def read(self, key: str) -> str:
        return self.records[key]

class AnotherKeyValueStore:
    def __init__(self):
        self._records = []
    def save(self, key: str, value: str) -> None:
        for i, (stored_key, _) in enumerate(self._records):
            if stored_key == key:
                self._records[i] = (key, value)
                return
        self._records.append((key, value))
    def read(self, key: str) -> str:
        for stored_key, value in self._records:
            if stored_key == key:
                return value
        raise KeyError(key)

def storage_contract_test(store_factory):
    store = store_factory()
    store.save("a", "first")
    assert store.read("a") == "first"
    store.save("a", "replacement")
    assert store.read("a") == "replacement"
    try:
        store.read("missing")
    except KeyError:
        pass
    else:
        raise AssertionError("Missing keys must raise KeyError")

for factory in (InMemoryKeyValueStore, AnotherKeyValueStore):
    storage_contract_test(factory)
print("Both store implementations satisfy the test contract")

## 11.2 Optional pytest version

If you install `pytest`, you can express contract checks using `@pytest.mark.parametrize` and share the same test function across factories. The equivalent code above uses plain `assert` so the notebook executes without extra packages.

# 12. Practical Exercises

Write code in new notebook cells or a separate project. These exercises are intentionally **not solved** so you can validate understanding independently.

## 12.1 Exercise 1 — SRP: User management

Start with a `UserManager` that validates user input, creates users, saves users, sends welcome emails and generates activity reports.

- [ ] Identify independent reasons to change.
- [ ] Refactor into coherent components.
- [ ] Keep business rules independent of the email provider.
- [ ] Write unit tests for user rules and integration tests for storage.

**Success:** changing the email provider does not change user business rules.

## 12.2 Exercise 2 — OCP: Discounts

Start with `if/elif` for percentage, fixed and seasonal discounts.

- [ ] Introduce a strategy only after identifying the variation.
- [ ] Add a new discount type without editing the calculation service.
- [ ] Test all strategies against their contracts.
- [ ] Explain whether a strategy is actually justified.

**Success:** new discount algorithms require new implementations, not repeated edits to stable discount consumers.

## 12.3 Exercise 3 — LSP: File storage

Model local, cloud and in-memory storage.

- [ ] Define precise `save`, `read` and `delete` contracts.
- [ ] Specify error behavior and consistency guarantees.
- [ ] Run the same tests against every compatible adapter.
- [ ] Explain any real differences requiring distinct contracts.

**Success:** compatible implementations can be substituted without hidden client-side exceptions.

## 12.4 Exercise 4 — ISP: Document management

An application supports reading, writing, deleting, searching and exporting documents. Some clients are read-only.

- [ ] Refactor a fat document interface into focused capabilities.
- [ ] Implement a read-only client and a full-access client.
- [ ] Ensure read-only tests need no write/delete/export mocks.

**Success:** each client sees only the operations it uses.

## 12.5 Exercise 5 — DIP: Notification providers

Design email, SMS and application notifications.

- [ ] Define a focused notification contract.
- [ ] Implement two concrete adapters.
- [ ] Inject the selected adapter at a composition root.
- [ ] Use a fake adapter for business-service unit tests.

**Success:** changing vendors does not rewrite business workflows.

## 12.6 Integrated capstone — Order management

Build a small order system with: order creation, validation, total calculation, multiple discounts, payments, persistence and customer notifications.

Suggested responsibilities (not a mandatory class hierarchy): `Order`, `OrderItem`, `OrderService`, `DiscountStrategy`, `PaymentGateway`, `OrderRepository`, `NotificationSender`.

**Deliverables:**

1. A short architecture diagram.
2. Two implementations for at least one genuine variation point.
3. One in-memory fake and one real infrastructure adapter.
4. Unit, contract and integration tests.
5. A design review showing at least **five** real design issues in a deliberately poorly structured baseline.
6. An Architecture Decision Record explaining one abstraction you **added** and one you deliberately **did not add**.

# 13. Mastery Checklist

## 13.1 SRP

- [ ] I can explain SRP through coherent reasons to change.
- [ ] I can distinguish SRP from the broader Separation of Concerns concept.
- [ ] I can separate independent business, persistence and communication concerns.
- [ ] I can identify excessive class fragmentation.
- [ ] I can test components in isolation.

## 13.2 OCP

- [ ] I can identify realistic variation points.
- [ ] I can replace repeated conditional branching with an appropriate strategy when justified.
- [ ] I can add new behavior through an existing contract.
- [ ] I understand why OCP does not ban modifications.
- [ ] I can balance extension against KISS and YAGNI.

## 13.3 LSP

- [ ] I understand behavioral subtyping.
- [ ] I can specify preconditions, postconditions and invariants.
- [ ] I can recognize incompatible failure modes and unsupported operations.
- [ ] I can write reusable contract tests.
- [ ] I can redesign misleading abstractions when business rules differ.

## 13.4 ISP

- [ ] I can identify fat interfaces.
- [ ] I can create focused Python `Protocol`s.
- [ ] I understand `Protocol` vs `ABC` and structural subtyping.
- [ ] I can eliminate unsupported methods from client contracts.
- [ ] I avoid unnecessary fragmentation.

## 13.5 DIP

- [ ] I can distinguish business policy from infrastructure detail.
- [ ] I can explain source-code dependency direction.
- [ ] I can define useful repository/service ports.
- [ ] I can distinguish DIP from Dependency Injection.
- [ ] I can use a composition root and test fakes.
- [ ] I understand why adapter integration tests are still needed.

## 13.6 Final assessment

- [ ] I have refactored a Python codebase using SOLID.
- [ ] I can identify at least five concrete design issues and their consequences.
- [ ] I can demonstrate behavior-preserving tests for my refactor.
- [ ] I can justify the abstractions I introduce.
- [ ] I can evaluate AI-generated code for violations **without** blindly forcing every principle into every class.

# 14. Recommended Learning Resources

## 14.1 Books

- **Clean Architecture — Robert C. Martin:** SOLID, boundaries, components and dependency direction.
- **Agile Software Development, Principles, Patterns, and Practices — Robert C. Martin:** OOP, SOLID, patterns and design examples.
- **Refactoring — Martin Fowler:** code smells and incremental, behavior-preserving design improvements.

## 14.2 Course mapping

**Course A — Software Architecture and Clean Code Design in OOP (Andrii Piatakha)**  
https://www.udemy.com/course/software-architecture-learnit/

Focus on SOLID, cohesion and coupling, KISS/YAGNI/DRY, GRASP, design patterns and layered architecture. Supplement the course with Python exercises and explicit behavioral-contract tests.

**Course B — Software Architecture & Design of Modern Large Scale Systems (Michael Pogrebinsky)**  
https://www.udemy.com/course/software-architecture-design-of-modern-large-scale-systems/

Take after code-level foundations. Focus on architecture requirements, quality attributes, distributed systems, scalability and end-to-end system design.

## 14.3 Python reference

- [Python `abc` module](https://docs.python.org/3/library/abc.html)
- [Python typing and `Protocol`](https://docs.python.org/3/library/typing.html)
- [Python dataclasses](https://docs.python.org/3/library/dataclasses.html)
- [Martin Fowler's articles](https://martinfowler.com/)
- [Refactoring.Guru](https://refactoring.guru/)

---

**Final takeaway:** The objective of SOLID is a system that is easier to understand, change, test and maintain. Measure success by better boundaries and reduced change cost—not by the number of classes or interfaces you produce.